# 00 — First principles of enterprise search & Q&A

> **Applied Labs** · **Domain**: RAG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/12_enterprise_search/00_first_principles.ipynb)

**Prerequisites**:
- Basic Python and NLP familiarity (embeddings, similarity)
- Understanding of text processing and information retrieval

**What you will learn**:
- Why keyword search fails for enterprise knowledge discovery
- The vocabulary mismatch problem and its quantifiable cost
- Why users want answers, not links — and what that changes architecturally
- How multi-source fragmentation degrades search quality
- Why freshness and access control are non-negotiable in enterprises
- The $8 B+ enterprise search market and the ROI of intelligent search

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "numpy>=1.24.0" "matplotlib>=3.7.0"

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set
from collections import Counter, defaultdict
import re, math, textwrap, json, time

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — Why keyword search fails

Enterprise search traditionally relies on **BM25** — a bag-of-words ranking function
that scores documents by **term frequency** and **inverse document frequency**.

The core problem: a user searching "how to handle customer refunds" expects the
company's *Return & Exchange Guidelines* document. But BM25 ranks documents by
keyword overlap, not semantic intent. The policy doc may never mention "refunds"
explicitly — it uses "returns", "exchanges", "merchandise credit".

Let's build a BM25 engine and expose this failure mode.

In [ ]:
@dataclass
class Document:
    """An enterprise knowledge-base document."""
    doc_id: str
    title: str
    source: str           # wiki, chat, docs, email
    content: str
    tags: List[str] = field(default_factory=list)

# Simulate a 12-document enterprise KB
CORPUS: List[Document] = [
    Document("D01", "Return & Exchange Guidelines",       "wiki",
             "All merchandise may be returned within 30 days. Exchanges follow the same window. "
             "Customers should present original receipt. Store credit is issued for items without receipt. "
             "Defective items qualify for full replacement regardless of time elapsed."),
    Document("D02", "Refund Processing Checklist",         "docs",
             "Step 1: Verify refund eligibility. Step 2: Log refund request in CRM. "
             "Step 3: Process refund via payment gateway. Step 4: Send confirmation email."),
    Document("D03", "Q3 Customer Satisfaction Survey",     "docs",
             "Survey results indicate 72% satisfaction with return and exchange process. "
             "Main complaints: slow processing time, unclear policy for digital goods."),
    Document("D04", "Slack: #support — refund escalation", "chat",
             "Hey team — customer wants a refund on a subscription. Checked the policy, "
             "subscription refunds go through finance. Loop in @jane."),
    Document("D05", "Employee PTO Policy 2024",            "wiki",
             "Full-time employees earn 15 days paid time off per year. Unused PTO rolls over "
             "up to 5 days. Requests must be submitted 2 weeks in advance via HR portal."),
    Document("D06", "Kubernetes Deployment Runbook",       "docs",
             "Deploy services to k8s cluster using Helm charts. Ensure namespace isolation. "
             "Run health checks post-deploy. Rollback procedure: helm rollback <release>."),
    Document("D07", "Product Spec: Widget Pro v2",         "docs",
             "Widget Pro v2 features: 20% faster processing, new REST API, improved dashboard. "
             "Target release: Q4 2024. Dependencies: auth service v3, billing service v2."),
    Document("D08", "Onboarding Checklist — Engineering",  "wiki",
             "Day 1: laptop setup, VPN access. Day 2: codebase walkthrough. "
             "Day 3: first PR. Week 1: shadow on-call rotation."),
    Document("D09", "Email: Vendor contract renewal",      "email",
             "Hi team, the AWS contract is up for renewal in March. Current spend is $48K/mo. "
             "Please review usage reports and flag any optimization opportunities."),
    Document("D10", "Support Ticket #4521 — Billing error","email",
             "Customer reports double charge on invoice #8832. Finance confirmed duplicate "
             "transaction. Refund issued and customer notified."),
    Document("D11", "Security Incident Response Plan",     "wiki",
             "On detection of a security incident: 1) contain, 2) investigate, "
             "3) remediate, 4) post-mortem. Escalation contacts in Appendix A."),
    Document("D12", "Meeting Notes: Product Roadmap Q4",   "docs",
             "Priorities: enterprise SSO, advanced analytics, API rate limiting. "
             "Stretch goal: AI-powered search for internal knowledge base."),
]


def tokenize(text: str) -> List[str]:
    """Simple whitespace + punctuation tokenizer."""
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25:
    """Minimal BM25 implementation for demonstration."""

    def __init__(self, documents: List[Document], k1: float = 1.5, b: float = 0.75):
        self.docs = documents
        self.k1 = k1
        self.b = b
        self.doc_tokens: List[List[str]] = [tokenize(d.title + " " + d.content) for d in documents]
        self.avgdl = np.mean([len(t) for t in self.doc_tokens])
        self.N = len(documents)
        # Build inverted index
        self.df: Dict[str, int] = defaultdict(int)
        for tokens in self.doc_tokens:
            for tok in set(tokens):
                self.df[tok] += 1

    def score(self, query: str) -> List[Tuple[float, Document]]:
        q_tokens = tokenize(query)
        scores: List[float] = []
        for i, doc_toks in enumerate(self.doc_tokens):
            tf_map = Counter(doc_toks)
            dl = len(doc_toks)
            s = 0.0
            for qt in q_tokens:
                tf = tf_map.get(qt, 0)
                df = self.df.get(qt, 0)
                idf = math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)
                num = tf * (self.k1 + 1)
                den = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                s += idf * num / den
            scores.append(s)
        ranked = sorted(zip(scores, self.docs), key=lambda x: -x[0])
        return ranked


bm25 = BM25(CORPUS)
query = "how to handle customer refunds"
results = bm25.score(query)

print(f"Query: '{query}'\n")
print(f"{'Rank':<6}{'Score':>8}  {'Doc ID':<8}{'Title'}")
print("-" * 70)
for rank, (score, doc) in enumerate(results[:5], 1):
    print(f"{rank:<6}{score:>8.3f}  {doc.doc_id:<8}{doc.title}")

# The BEST document for this query is D01 — but it may not rank #1
target_rank = next(i for i, (_, d) in enumerate(results, 1) if d.doc_id == "D01")
print(f"\n⚠  'Return & Exchange Guidelines' (D01) ranked #{target_rank}")
print("   BM25 rewards keyword overlap, not semantic relevance.")
print(f"   Precision@5 for correct doc: {1 / target_rank:.2f}")

## Section 2 — The vocabulary mismatch problem

The failure above isn't a bug — it's structural. In enterprises, people use
**synonyms**, **abbreviations**, **jargon**, and **acronyms** interchangeably:

| User says | Document says | Match? |
|-----------|---------------|--------|
| PTO | paid time off | ✗ |
| k8s | Kubernetes | ✗ |
| refund | return/exchange | ✗ |
| onboarding | new hire setup | ✗ |

This is the **vocabulary mismatch problem** — the #1 reason keyword search
fails in enterprise settings. Let's quantify it.

In [ ]:
# Vocabulary mismatch examples
SYNONYM_PAIRS: List[Tuple[str, str]] = [
    ("PTO", "paid time off"),
    ("k8s", "Kubernetes"),
    ("refund", "return and exchange"),
    ("onboarding", "new hire setup"),
    ("SSO", "single sign-on"),
    ("SLA", "service level agreement"),
    ("PR", "pull request"),
    ("OKR", "objective and key result"),
    ("EOD", "end of day"),
    ("LGTM", "looks good to me"),
]


def token_overlap(a: str, b: str) -> float:
    """Jaccard similarity between token sets."""
    set_a = set(tokenize(a))
    set_b = set(tokenize(b))
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


print(f"{'Query term':<20} {'Document term':<25} {'Token overlap'}")
print("-" * 60)
overlaps: List[float] = []
for user_term, doc_term in SYNONYM_PAIRS:
    ov = token_overlap(user_term, doc_term)
    overlaps.append(ov)
    print(f"{user_term:<20} {doc_term:<25} {ov:.2f}")

avg_overlap = np.mean(overlaps)
print(f"\nAverage token overlap: {avg_overlap:.2f}")
print(f"Zero-overlap pairs:   {sum(1 for o in overlaps if o == 0.0)} / {len(overlaps)}")
print("\n→ Keyword search is structurally blind to most enterprise jargon.")

## Section 3 — From retrieval to answers

The deeper insight: even when keyword search *does* return the right document,
the user still has to **read 10 results**, **open each link**, and **extract the
answer manually**. What they actually want is a **direct answer** with sources.

Let's compare the user experience: 10 blue links vs. a synthesized answer.

In [ ]:
# Simulate: search results page vs. synthesized answer
query = "What is the PTO policy for full-time employees?"

# Traditional search: return ranked document titles
bm25_results = bm25.score(query)[:10]

print("═" * 70)
print("TRADITIONAL SEARCH RESULTS — 10 blue links")
print("═" * 70)
for i, (score, doc) in enumerate(bm25_results, 1):
    snippet = doc.content[:80] + "..."
    print(f"  {i}. [{doc.source}] {doc.title}")
    print(f"     {snippet}")
    print()

# Synthesized answer (simulated — in 02_build.ipynb we'll use an LLM)
synthesized = (
    "Full-time employees earn **15 days** of paid time off per year. "
    "Unused PTO rolls over up to **5 days**. Requests must be submitted "
    "**2 weeks in advance** via the HR portal."
)
print("═" * 70)
print("SYNTHESIZED ANSWER — with citation")
print("═" * 70)
print(f"  Q: {query}\n")
print(f"  A: {synthesized}")
print(f"  📄 Source: Employee PTO Policy 2024 (D05, wiki)")

# Time-to-answer comparison
time_links = 3.5    # minutes — user reads ~3 docs
time_synth = 0.08   # minutes — ~5 seconds

print(f"\n⏱  Estimated time-to-answer:")
print(f"   10 blue links : {time_links:.1f} min (open ~3 docs, scan, synthesize mentally)")
print(f"   RAG answer    : {time_synth:.2f} min (~5 sec)")
print(f"   Speedup       : {time_links / time_synth:.0f}×")

## Section 4 — Multi-source challenge

Enterprise knowledge is **fragmented** across systems: Confluence wikis, Slack
threads, Google Drive docs, email archives, Jira tickets, Notion pages.

Each source has its own:
- Format (HTML, Markdown, plain text, PDF)
- Access model (public, team, individual)
- Update cadence (real-time chat → quarterly policies)

A single question may require synthesizing information from **3+ sources**.
Let's build a fragmentation analysis.

In [ ]:
# Analyze source distribution in our corpus
source_docs: Dict[str, List[str]] = defaultdict(list)
for doc in CORPUS:
    source_docs[doc.source].append(doc.doc_id)

print("Knowledge distribution across sources:")
print("-" * 45)
for source, doc_ids in sorted(source_docs.items()):
    bar = "█" * len(doc_ids)
    print(f"  {source:<8} {bar} ({len(doc_ids)} docs): {', '.join(doc_ids)}")

# Multi-source queries — answers require information from multiple sources
MULTI_SOURCE_QUERIES: List[Dict[str, any]] = [
    {
        "query": "What happened with the customer billing error?",
        "relevant_docs": ["D02", "D04", "D10"],
        "sources_needed": {"docs", "chat", "email"},
    },
    {
        "query": "What are the Q4 product priorities and timeline?",
        "relevant_docs": ["D07", "D12"],
        "sources_needed": {"docs"},
    },
    {
        "query": "How do we deploy and what security measures are in place?",
        "relevant_docs": ["D06", "D11"],
        "sources_needed": {"docs", "wiki"},
    },
]

print("\nMulti-source query analysis:")
print("-" * 60)
for q_info in MULTI_SOURCE_QUERIES:
    print(f"  Q: {q_info['query']}")
    print(f"     Relevant docs  : {q_info['relevant_docs']}")
    print(f"     Sources needed : {q_info['sources_needed']}")
    single_source_coverage = 1 / len(q_info["sources_needed"])
    print(f"     Single-source coverage: {single_source_coverage:.0%}")
    print()

print("→ Siloed search (one source at a time) misses critical context.")

## Section 5 — Freshness and access control

Two enterprise-critical requirements that consumer search ignores:

1. **Freshness** — Policies update, Slack messages obsolete old decisions.
   Returning a 2-year-old security policy when a new one exists is dangerous.

2. **Access control** — A search engine that surfaces salary data to all
   employees, or shares M&A plans with interns, creates legal liability.

Let's build scenarios that expose both failure modes.

In [ ]:
from datetime import datetime, timedelta

@dataclass
class ACLDocument:
    """Document with freshness and access metadata."""
    doc_id: str
    title: str
    content: str
    updated: datetime
    acl_groups: Set[str]       # groups that may access this doc
    superseded_by: Optional[str] = None  # newer version, if any

# Freshness scenario
now = datetime.now()
versioned_docs: List[ACLDocument] = [
    ACLDocument("SEC-v1", "Security Policy v1", "Use MD5 for hashing passwords.",
                now - timedelta(days=900), {"all_employees"}, superseded_by="SEC-v2"),
    ACLDocument("SEC-v2", "Security Policy v2", "Use bcrypt with cost factor ≥ 12 for hashing.",
                now - timedelta(days=30), {"all_employees"}),
    ACLDocument("SAL-01", "2024 Salary Bands — Engineering", "L3: $120-160K, L4: $150-200K, L5: $190-260K.",
                now - timedelta(days=60), {"hr_team", "exec_team"}),
    ACLDocument("MA-01",  "Project Phoenix — Acquisition Target", "Target: Acme Corp. Offer: $45M.",
                now - timedelta(days=10), {"exec_team"}),
]

# Freshness problem
print("═══ FRESHNESS PROBLEM ═══")
print(f"Query: 'password hashing policy'\n")
for doc in versioned_docs[:2]:
    age_days = (now - doc.updated).days
    status = "⚠ STALE" if doc.superseded_by else "✓ CURRENT"
    print(f"  [{doc.doc_id}] {doc.title}  (age: {age_days}d)  {status}")
    print(f"    Content: {doc.content}")
print("\n  Without freshness tracking, BM25 may rank the STALE v1 higher")
print("  (more keyword matches) → dangerous security guidance.\n")

# Access control problem
print("═══ ACCESS CONTROL PROBLEM ═══")
user_groups = {"all_employees", "engineering"}
print(f"User groups: {user_groups}\n")
for doc in versioned_docs[2:]:
    has_access = bool(user_groups & doc.acl_groups)
    status = "✓ ALLOWED" if has_access else "✗ BLOCKED"
    print(f"  [{doc.doc_id}] {doc.title}")
    print(f"    Required groups: {doc.acl_groups}  → {status}")
print("\n  A search engine without ACL enforcement leaks sensitive information.")

## Section 6 — Market analysis

Enterprise search is a **$8 B+** market growing at 10 %+ CAGR, driven by:
- Remote work → more digital knowledge, harder to find
- Regulatory pressure → must retrieve compliance docs quickly
- AI capabilities → LLMs finally make semantic search + synthesis feasible

Key players: **Glean** ($4.6B valuation), **Elastic**, **Coveo**, **Lucidworks**.

Let's quantify the productivity cost of poor search and build an ROI model.

In [ ]:
# Enterprise search ROI calculator
print("═══ ENTERPRISE SEARCH ROI CALCULATOR ═══\n")

# Industry benchmarks
EMPLOYEES = 5_000
AVG_SALARY_ANNUAL = 95_000          # USD
HOURS_PER_YEAR = 2_080
HOURLY_COST = AVG_SALARY_ANNUAL / HOURS_PER_YEAR

# McKinsey: knowledge workers spend 19% of time searching for information
SEARCH_TIME_PCT = 0.19
SEARCH_HOURS_PER_YEAR = HOURS_PER_YEAR * SEARCH_TIME_PCT

# AI search reduces search time by 35-50% (conservative: 35%)
EFFICIENCY_GAIN = 0.35

hours_saved_per_employee = SEARCH_HOURS_PER_YEAR * EFFICIENCY_GAIN
total_hours_saved = hours_saved_per_employee * EMPLOYEES
dollar_savings = total_hours_saved * HOURLY_COST

print(f"Company size:             {EMPLOYEES:,} employees")
print(f"Avg hourly cost:          ${HOURLY_COST:.2f}")
print(f"Search time (current):    {SEARCH_HOURS_PER_YEAR:.0f} hrs/employee/year ({SEARCH_TIME_PCT:.0%})")
print(f"Efficiency gain:          {EFFICIENCY_GAIN:.0%}")
print(f"Hours saved/employee:     {hours_saved_per_employee:.0f} hrs/year")
print(f"Total hours saved:        {total_hours_saved:,.0f} hrs/year")
print(f"Annual savings:           ${dollar_savings:,.0f}")
print()

# Cost comparison
TOOL_COST_PER_SEAT = 15  # $/month
ANNUAL_TOOL_COST = TOOL_COST_PER_SEAT * 12 * EMPLOYEES

print(f"Tool cost ($15/seat/mo):  ${ANNUAL_TOOL_COST:,.0f}/year")
print(f"Net ROI:                  ${dollar_savings - ANNUAL_TOOL_COST:,.0f}/year")
print(f"ROI multiple:             {dollar_savings / ANNUAL_TOOL_COST:.1f}×")

# Visualization
categories = ["Search\ntime cost", "AI search\nsavings", "Tool cost", "Net ROI"]
values = [
    SEARCH_HOURS_PER_YEAR * HOURLY_COST * EMPLOYEES,
    dollar_savings,
    ANNUAL_TOOL_COST,
    dollar_savings - ANNUAL_TOOL_COST,
]
colors = ["#e74c3c", "#2ecc71", "#f39c12", "#3498db"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(categories, [v / 1e6 for v in values], color=colors, edgecolor="white", linewidth=1.2)
ax.set_ylabel("$ Millions")
ax.set_title("Enterprise Search ROI — 5,000-Employee Company")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f"${val / 1e6:.1f}M", ha="center", va="bottom", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 🏋️ Exercise 1 — Extend the vocabulary mismatch analysis

Add **5 more synonym pairs** commonly found in your own domain (e.g., DevOps,
healthcare, finance). Recompute token overlaps and find any pair with non-zero
Jaccard similarity — explain why it partially matches.

In [ ]:
# YOUR CODE HERE
# Hint: add to SYNONYM_PAIRS and rerun the analysis
# Example: ("CI/CD", "continuous integration and delivery")

## 🏋️ Exercise 2 — BM25 failure cases

Write **3 queries** where BM25 returns the wrong #1 result on our CORPUS.
For each, explain *why* BM25 fails and what a semantic system would do differently.

In [ ]:
# YOUR CODE HERE
# Hint: think about queries that use different words than the target document
# test_queries = ["...", "...", "..."]
# for q in test_queries:
#     results = bm25.score(q)
#     print(f"Query: {q}")
#     print(f"  BM25 #1: {results[0][1].title}")
#     print(f"  Expected: ???")

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | BM25 keyword search fails when users and documents use **different vocabulary** for the same concept |
| 2 | The **vocabulary mismatch problem** is structural — synonyms, abbreviations, and jargon cause zero token overlap |
| 3 | Users want **answers with citations**, not 10 blue links — RAG reduces time-to-answer by 40×+ |
| 4 | Enterprise knowledge is **fragmented** across 5–10 systems — single-source search misses critical context |
| 5 | **Freshness** and **access control** are non-negotiable — stale results are dangerous, leaked data is a liability |
| 6 | Enterprise search is an **$8 B+ market** with measurable ROI: 8–10× return on tool cost |

## What's Next

In **01_architecture.ipynb**, we design the full enterprise search pipeline:
connectors → chunking → embedding → hybrid retrieval → reranking → generation
with citations and access control.

## References

1. Robertson, S. & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. Foundations and Trends in Information Retrieval.
2. McKinsey Global Institute (2012). *The Social Economy: Unlocking Value and Productivity Through Social Technologies* — 19% of work time spent searching for information.
3. Karpukhin, V. et al. (2020). *Dense Passage Retrieval for Open-Domain Question Answering*. EMNLP 2020.
4. Glean (2024). Enterprise AI search platform. https://www.glean.com